In [3]:
# 1. INSTALL DEPENDENCIES (RUN ONCE)
!pip install faiss-cpu sentence-transformers groq

# 2. LOAD GROQ API KEY FROM COLAB SECRETS
import os
from google.colab import userdata

os.environ["api_key"] = userdata.get("api_key")

# 3. IMPORTS
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq

# 4. INITIALIZE GROQ CLIENT
client = Groq(api_key=os.environ["api_key"])

# 5. SYNTHETIC DATASET (KNOWLEDGE BASE)
documents = [
    "Refund policy: Students can request a full refund within 7 days of enrollment. After 7 days, no refund is allowed.",
    "Course deadline: All assignments must be submitted before the course end date. Late submissions are not accepted.",
    "Lecture: Machine Learning includes supervised and unsupervised learning techniques.",
    "Lecture: Neural Networks are inspired by the human brain and are a subset of machine learning.",
    "FAQ: Students can access course materials anytime after enrollment.",
    "FAQ: Certificates are issued after completing all assignments and quizzes."
]

# 6. CHUNKING
def chunk_text(text, chunk_size=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_text(doc))

# 7. EMBEDDINGS
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(all_chunks)

# 8. VECTOR STORE (FAISS)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# 9. RETRIEVAL FUNCTION
def retrieve_context(query, k=3):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)

    results = [all_chunks[i] for i in indices[0]]
    return "\n".join(results)

# 10. TOOLS (SIMULATED APIs)
def get_student_status(student_id):
    return f"Student {student_id} is enrolled and has completed 75% of the course."

def get_assignment_deadlines():
    return "Assignment 1: 2026-04-01, Assignment 2: 2026-04-10"

# 11. AGENT ROUTER
def route_query(query):
    q = query.lower()

    if "my" in q and ("status" in q or "progress" in q):
        return "student_tool"

    elif "deadline" in q:
        return "deadline_tool"

    else:
        return "rag"

# 12. GROQ LLM FUNCTION (UPDATED MODEL)
def ask_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

# 13. MAIN SYSTEM (END-TO-END)
def edtech_assistant(query, student_id="123"):

    route = route_query(query)

    if route == "student_tool":
        tool_output = get_student_status(student_id)
        prompt = f"""
        Answer the question using the data below.

        Data:
        {tool_output}

        Question: {query}
        """

    elif route == "deadline_tool":
        tool_output = get_assignment_deadlines()
        prompt = f"""
        Answer the question using the data below.

        Data:
        {tool_output}

        Question: {query}
        """

    else:
        context = retrieve_context(query)

        # Reliability: avoid hallucination
        if context.strip() == "":
            return "I don't know"

        prompt = f"""
        You are a reliable AI assistant.

        Rules:
        - Answer ONLY from the provided context
        - If not found, say "I don't know"
        - Do NOT hallucinate

        Context:
        {context}

        Question:
        {query}
        """

    return ask_llm(prompt)

# 14. TESTING
if __name__ == "__main__":

    queries = [
        "What is the refund policy?",
        "Explain neural networks",
        "What is my progress?",
        "When is assignment 1 deadline?",
        "Can I access materials after enrollment?"
    ]

    for q in queries:
        print("Q:", q)
        print("A:", edtech_assistant(q))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 10.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q: What is the refund policy?
A: Students can request a full refund within 7 days of enrollment. After 7 days, no refund is allowed.
Q: Explain neural networks
A: Neural Networks are inspired by the human brain and are a subset of machine learning.
Q: What is my progress?
A: You have completed 75% of the course.
Q: When is assignment 1 deadline?
A: The deadline for Assignment 1 is 2026-04-01.
Q: Can I access materials after enrollment?
A: Yes, students can access course materials anytime after enrollment.
